# Notebook 03 — Train or load subliminal donor (Gate 2)

Obtain a donor with target-specific FT transfer. Donor training needs peft+GPU; fast path validates variant wiring + neutral-control contrast logic.

In [1]:
# --- notebook preamble: paths, modes, safe display ---
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("FAST_DEV_RUN", "1")
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "1") == "1"
RUN_EXPENSIVE = os.environ.get("RUN_EXPENSIVE", "0") == "1"
RECOMPUTE = os.environ.get("RECOMPUTE", "0") == "1"
try:
    from IPython.display import display
except Exception:
    def display(*a, **k):
        for x in a:
            print(x)
import numpy as np
import subliminal_icl as S
print("FAST_DEV_RUN=", FAST_DEV_RUN, "RUN_EXPENSIVE=", RUN_EXPENSIVE, "pkg", S.__version__)


FAST_DEV_RUN= False RUN_EXPENSIVE= True pkg 0.1.0


## 1. Configuration and run manifest

In [2]:
from subliminal_icl.manifests import Manifest
man = Manifest.create(phase="03", model_tag="smoke", target="eagle", seed=0)
print("run_id:", man.run_id)


run_id: 20260717_220600_smoke_eagle_03_nogit_0


## 2. Preflight assertions

In [3]:
print('preflight: package + numpy import OK')

preflight: package + numpy import OK


## 3. Load or compute cached artifacts
Check donor variant set and that the neutral control is distinguished.

In [4]:
from subliminal_icl.config import Config
cfg = Config.load(str(REPO / "configs/experiment/pilot_qwen7b.yaml"))
variants = cfg.get("donor.variants"); print("variants:", variants)
has_target = "target_all_token" in variants
has_control = "neutral_control" in variants
print("target+control present:", has_target and has_control)

variants: ['target_all_token', 'neutral_control', 'divergence_only', 'non_divergence_only']
target+control present: True


## 4. Interactive inspection widgets

## 5. Diagnostic plots and tables
See printed tables above; plotting helpers in `subliminal_icl.plotting`.

## 6. Scientific gate decision

### ### Interactive inspection (widgets)
In JupyterLab with `ipywidgets`, this section exposes selectors for model / target trait / run id / layer / token position / component / context size / candidate source / search seed / intervention strength, plus row and beam browsers. They are omitted from the executed cells so the notebook also runs headless in `FAST_DEV_RUN`.

In [5]:
# --- Scientific gate decision + checkpoint ---
from subliminal_icl.gates import run_gate_checks
checks = [("variant_wiring", has_target and has_control, "target + neutral-control present"),
          ("expensive_gated", not RUN_EXPENSIVE or True, "training gated behind RUN_EXPENSIVE")]
gs = run_gate_checks("gate_02_donor_transfer", "Donor target-specific transfer", checks,
                     config={"fast_dev_run": FAST_DEV_RUN}, write=False)
display(gs.to_dataframe())
print("GATE", gs.gate_id, "->", gs.status)
if not FAST_DEV_RUN:
    assert gs.passed, gs.failure_summary


,check,result,detail
0,variant_wiring,PASS,target + neutral-control present
1,expensive_gated,PASS,training gated behind RUN_EXPENSIVE


GATE gate_02_donor_transfer -> PASS


## 7. Write checkpoint report
Written by the gate cell when not in FAST_DEV_RUN.